# 05 · Files for the web app

The last notebook. It makes everything the web app (`app/`) needs from the results of notebooks 01–04, in a folder
`app/` laid out like the project's `app/` folder, so it can be copied over it:

| File | What it is |
|---|---|
| `app/models/model.pt` | the app's model: MobileNetV2 from the row with the best **validation** macro-F1, with its "not sure" threshold |
| `app/frontend/public/library/<class>.jpg` | one typical training photo per class, for the disease library |
| `app/frontend/public/gan/<class>-N.jpg` | a few GAN images per class, for the About page |
| `app/frontend/src/data/results.js` | the comparison table (test split, mean of 3 seeds) for the About page |
| `app/frontend/src/data/gan.js` | the GAN facts for the About page (kimg, KID, how many images were kept) |
| `app_files.zip` | all of the above in one file |

**The "not sure" threshold.** Below this confidence the app answers "Not sure, please retake" instead of a disease.
It is chosen on the validation split: the threshold below which 1% of the *correct* validation answers fall (so 99% of
the correct answers stay above it), kept between 0.5 and 0.95. It is stored in `model.pt`, and the app uses it.

**Final export after notebook 04 session 2**

The default `REQUIRE_COMPLETE_COMPARISON = True` checks that all 60 runs are present (12 baseline runs plus
48 retraining runs). It stops with a list of missing runs if an older or incomplete output is attached. Set it to
`False` only when deliberately making an earlier, partial export.

**How to run it on Kaggle with your uploaded datasets**

1. Import this updated `05_app_files.ipynb`.
2. Attach the same **PlantVillage** dataset used before.
3. Attach your existing **kaggle-retrain-input** dataset. It already contains the frozen `data/splits.csv`, adjacent
   `classes.json`, `results/02_final_runs.json` and `synthetic/` (manifest plus all JPEGs).
4. Attach notebook 04's **completed session-2 output**, preserving sibling `results/` and `models/` folders.
   Its `results/04_runs.json` has 48 runs. Attach only the newest output when possible.
5. For GAN quality facts, upload just **`03_summary.json`** from `results-03-gan-generate/` as another input.
   The full generation output and GAN checkpoints are unnecessary when `kaggle-retrain-input` is attached.
   The summary is optional: without it the notebook prints a warning, exports the GAN images and counts, and
   leaves the unavailable quality fields empty. A supplied summary is checked against the split and image counts.
6. Keep `APP_ARCH = 'mobilenet_v2'`, `APP_ROW = None`, and `REQUIRE_COMPLETE_COMPARISON = True`.
   The reviewed session-2 results select **row C (oversampling), seed 2, best epoch 15** on validation scores.
7. **Settings:** Accelerator **None** is enough (roughly 10 minutes on CPU); Internet is not needed.
   Use **Save Version → Save & Run All (Commit)**.

Alternatively, attach the original outputs of notebooks 01, 02 (final), 03_gan_generate and 04 (session 2).
The notebook discovers the same files automatically. If row A is selected, its baseline checkpoint is also needed.

**Expected output:** all 60 runs found; MobileNetV2 row C, seed 2; `results.js: 20 rows`; and
`GAN images: 36 copied for 9 classes`. Download `app_files.zip` and `results/`. Save the latter locally as
`results-05-session2/` for review. After checking, unzip `app_files.zip` in the project root, replacing the included
files in `app/`, and start `run-app.bat`.


In [ ]:
import os, json, time, shutil, hashlib, fnmatch, zipfile
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torchvision import models

APP_ARCH = 'mobilenet_v2'   # the app's model: small and fast. 'resnet50' if a larger model is clearly better
APP_ROW = None              # None: the row (A-E) with the best mean validation macro-F1 for APP_ARCH; or e.g. 'D'
REQUIRE_COMPLETE_COMPARISON = True  # final export: all A-E rows, 4 models and 3 seeds must be present
KEEP_CORRECT = 0.99         # the threshold keeps this share of the correct validation answers above it
THRESHOLD_LIMITS = (0.5, 0.95)
GAN_SAMPLES = 4             # GAN images per class on the About page
LIBRARY_PER_CLASS = 300     # training photos per class looked at when choosing the library photo
LIBRARY_PICKS = {}          # to choose a library photo yourself: {'Tomato___Leaf_Mold': 'Tomato___Leaf_Mold/<file>.JPG'}
SPLIT_FINGERPRINT = '7cf95c207559517a'
SEED = 0

# The page names of the classes in the app (app/frontend/src/data/diseases.js)
SLUGS = {
    'Tomato___Bacterial_spot': 'bacterial-spot', 'Tomato___Early_blight': 'early-blight',
    'Tomato___Late_blight': 'late-blight', 'Tomato___Leaf_Mold': 'leaf-mold',
    'Tomato___Septoria_leaf_spot': 'septoria-leaf-spot', 'Tomato___Spider_mites Two-spotted_spider_mite': 'spider-mites',
    'Tomato___Target_Spot': 'target-spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'yellow-leaf-curl-virus',
    'Tomato___Tomato_mosaic_virus': 'mosaic-virus', 'Tomato___healthy': 'healthy',
}
ROW_NAMES = {'A': 'Real only', 'B': 'Real + ordinary augmentation', 'C': 'Real, small classes repeated',
             'D': 'Real + GAN', 'E': 'Real + GAN + ordinary augmentation'}
NAMES = {'custom_cnn': 'Custom CNN', 'vgg16': 'VGG16', 'resnet50': 'ResNet50', 'mobilenet_v2': 'MobileNetV2'}
MODELS = ['custom_cnn', 'vgg16', 'resnet50', 'mobilenet_v2']   # the order of the About page table

INPUT_ROOT = Path(os.environ.get('INPUT_ROOT', '/kaggle/input'))
OUT_DIR = Path(os.environ.get('OUT_DIR', '/kaggle/working'))
APP_OUT = OUT_DIR / 'app'
RESULTS_OUT = OUT_DIR / 'results'
for d in (APP_OUT / 'models', APP_OUT / 'frontend/public/library', APP_OUT / 'frontend/src/data', RESULTS_OUT):
    d.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

SURFACE, INK, INK_2, GRID, BLUE, ORANGE = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#2a78d6', '#eb6834'
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'text.color': INK, 'axes.labelcolor': INK_2, 'xtick.color': INK_2, 'ytick.color': INK_2,
    'axes.edgecolor': GRID, 'axes.linewidth': 1, 'grid.color': GRID, 'grid.linewidth': 1,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

## 1. The inputs

In [ ]:
def find_files(root, pattern):
    """Files under root whose name matches pattern. Class folders (names with ___) only hold images, so they are skipped."""
    found = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        dirnames[:] = [d for d in dirnames if '___' not in d]
        found += [Path(dirpath) / f for f in sorted(filenames) if fnmatch.fnmatch(f, pattern)]
    return found

def find_color_dir(root):
    for dirpath, dirnames, _ in os.walk(root):
        dirnames.sort()
        if Path(dirpath).name.lower() == 'color' and any(d.startswith('Tomato') for d in dirnames):
            return Path(dirpath)
        dirnames[:] = [d for d in dirnames if '___' not in d]
    raise FileNotFoundError(f'No PlantVillage "color" folder under {root}. '
                            'Add the Kaggle dataset abdallahalidev/plantvillage-dataset as input.')

def fingerprint(splits):
    return hashlib.sha256('\n'.join(sorted(splits['path'] + ',' + splits['split'])).encode()).hexdigest()[:16]

def newest(files):
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

found = {f: pd.read_csv(f) for f in find_files(INPUT_ROOT, 'splits.csv')}
final_split = [f for f, s in found.items() if fingerprint(s) == SPLIT_FINGERPRINT]
if not final_split:
    raise FileNotFoundError('The final splits.csv of notebook 01 was not found. Add Input → Your Work → Notebooks → '
                            'notebook 01.')
SPLITS_CSV = final_split[0]
splits = found[SPLITS_CSV]
classes = json.loads((SPLITS_CSV.parent / 'classes.json').read_text())
K = len(classes)
COLOR_DIR = find_color_dir(INPUT_ROOT)
assert set(SLUGS) == {c['folder'] for c in classes}, 'the class folders do not match the app'

def read_json(file):
    return json.loads(file.read_text(encoding='utf-8-sig'))

def checked_state(file, notebook):
    state = read_json(file)
    settings = state.get('settings', {})
    if settings.get('run') != 'final' or settings.get('scores_on') != 'test':
        raise ValueError(f'{file} is not a final run scored on test. Attach the final notebook {notebook} output.')
    if settings.get('split_fingerprint') != SPLIT_FINGERPRINT:
        raise ValueError(f'{file} uses a different split. Attach the frozen project inputs.')
    return state

# Row A can come from the combined retrain dataset or notebook 02's final output.
a_file = newest(find_files(INPUT_ROOT, '02_final_runs.json'))
if a_file is None:
    raise FileNotFoundError('02_final_runs.json was not found. Attach kaggle-retrain-input or notebook 02 final output.')
a_state = checked_state(a_file, '02')
runs = [dict(r, row='A') for r in a_state['runs']]

# Choose the most complete 04 output; its sibling models/ is preferred for the checkpoint.
b_files = find_files(INPUT_ROOT, '04_runs.json')
b_file = max(b_files, key=lambda f: len(read_json(f)['runs']), default=None)
b_state = checked_state(b_file, '04') if b_file else None
if b_state is not None:
    for setting in ('img_size', 'normalisation', 'seeds', 'max_epochs'):
        if b_state['settings'].get(setting) != a_state['settings'].get(setting):
            raise ValueError(f'Notebook 04 and baseline settings differ: {setting}. Attach matching final outputs.')
    runs += b_state['runs']

SEEDS = [0, 1, 2]
SEEDS_PER_ROW = len(SEEDS)
run_key = lambda r: (r['row'], r['model'], r['seed'])
keys = [run_key(r) for r in runs]
expected = {(row, arch, seed) for row in ROW_NAMES for arch in MODELS for seed in SEEDS}
if len(keys) != len(set(keys)):
    raise ValueError('Duplicate row/model/seed entries in the attached results. Attach an unmodified final output.')
unexpected = set(keys) - expected
if unexpected:
    raise ValueError(f'Unexpected run keys in the attached results: {sorted(unexpected)}')
missing = sorted(expected - set(keys))
if REQUIRE_COMPLETE_COMPARISON and missing:
    detail = ', '.join(f'{row}/{arch}/seed {seed}' for row, arch, seed in missing)
    raise ValueError(f'Final export needs all 60 runs; found {len(keys)}. Missing: {detail}. '
                     'Attach notebook 04 session-2 results/ and models/. '
                     'Use REQUIRE_COMPLETE_COMPARISON = False only for a deliberate partial export.')
if any('val' not in r or 'test' not in r for r in runs):
    raise ValueError('Every final run must contain both validation and test metrics.')
print('Row A:  ', a_file)
print('Rows B-E:', b_file or 'notebook 04 not attached')
print(f'Comparison: {len(keys)}/60 runs; {len(keys) // SEEDS_PER_ROW} complete row/model groups'
      if not missing else f'Partial comparison: {len(keys)}/60 runs')

# A separate uploaded 03_summary.json works with synthetic/ from kaggle-retrain-input.
synth_csv = newest(find_files(INPUT_ROOT, 'synthetic.csv'))
gan_summary_file = newest(find_files(INPUT_ROOT, '03_summary.json'))
synthetic = pd.read_csv(synth_csv) if synth_csv is not None else None
summary = read_json(gan_summary_file) if gan_summary_file is not None else {}
if REQUIRE_COMPLETE_COMPARISON and synthetic is None:
    raise FileNotFoundError('Final export needs synthetic/synthetic.csv and its JPEGs. Attach kaggle-retrain-input.')
if synthetic is not None:
    if synthetic['path'].duplicated().any():
        raise ValueError('The synthetic manifest contains duplicate image paths.')
    for c in classes:
        images = synthetic[synthetic['label'] == c['label']]
        if not images['class_folder'].eq(c['folder']).all() or not images['class_name'].eq(c['name']).all():
            raise ValueError(f'Synthetic label/class mismatch: {c["name"]}')
    if not synthetic['label'].isin([c['label'] for c in classes]).all():
        raise ValueError('The synthetic manifest has unknown class labels.')
    if b_state is not None and len(synthetic) != b_state['settings'].get('synthetic_images'):
        raise ValueError('Synthetic image count differs from notebook 04. Attach the same generation dataset.')
    if gan_summary_file is not None:
        if summary.get('split_fingerprint') != SPLIT_FINGERPRINT or summary.get('kept_images') != len(synthetic):
            raise ValueError('03_summary.json does not match the split or synthetic image count. Attach the matching summary.')
        facts = {r['class']: r for r in summary.get('per_class', [])}
        for c in classes:
            count = int((synthetic['label'] == c['label']).sum())
            real_count = int(((splits['split'] == 'train') & (splits['label'] == c['label'])).sum())
            fact = facts.get(c['name'], {})
            if count and (fact.get('kept') != count or fact.get('train images') != real_count):
                raise ValueError(f'03_summary.json class counts differ for {c["name"]}. Attach the matching summary.')
    else:
        print('WARNING: 03_summary.json is missing. GAN images/counts will be exported; '
              'training progress, KID and label-check facts will be unavailable. '
              'Upload just this JSON as an additional input for complete GAN facts.')
print('GAN images: ', synth_csv or 'not attached')
print('GAN summary:', gan_summary_file or 'not attached (quality facts unavailable)')

def group(row, arch, split='test'):
    return [r for r in runs if r['row'] == row and r['model'] == arch and split in r]


## 2. The app's model

Chosen on the **validation** split, as the rules say: for `APP_ARCH`, the row with the highest mean validation
macro-F1 over its 3 seeds. The model file of that row holds its best seed (also chosen on validation). The test
numbers are printed only to be shown on the About page.

In [ ]:
if APP_ARCH not in NAMES or (APP_ROW is not None and APP_ROW not in ROW_NAMES):
    raise ValueError('Choose a supported APP_ARCH and APP_ROW (None or A-E).')

table = []
for row in ROW_NAMES:
    for arch in NAMES:
        rs = group(row, arch)
        if {r['seed'] for r in rs} == set(SEEDS):
            table.append({'row': row, 'model': NAMES[arch], 'arch': arch,
                          'val macro-F1 %': 100 * np.mean([r['val']['macro_f1'] for r in rs]),
                          'test macro-F1 %': 100 * np.mean([r['test']['macro_f1'] for r in rs]),
                          'test accuracy %': 100 * np.mean([r['test']['accuracy'] for r in rs])})
table = pd.DataFrame(table)
print('Rows with all 3 seeds finished (validation decides; test only for information):\n')
print(table.pivot(index='row', columns='model', values='val macro-F1 %').round(2).to_string())

candidates = table[table['arch'] == APP_ARCH]
if candidates.empty:
    raise ValueError(f'No complete three-seed results for {APP_ARCH}. Attach matching final results.')
if APP_ROW is None:
    APP_ROW = candidates.sort_values('val macro-F1 %', ascending=False)['row'].iloc[0]
print(f'\nApp model: {NAMES[APP_ARCH]}, row {APP_ROW} ({ROW_NAMES[APP_ROW]})')

if APP_ROW not in set(candidates['row']):
    raise ValueError(f'Row {APP_ROW} has no complete three-seed results for {APP_ARCH}.')
run = max(group(APP_ROW, APP_ARCH), key=lambda r: (r['val']['macro_f1'], -r['val_loss']))
model_name = f'{APP_ARCH}.pt' if APP_ROW == 'A' else f'04_{APP_ROW}_{APP_ARCH}.pt'
source_file = a_file if APP_ROW == 'A' else b_file
preferred_model = source_file.parent.parent / 'models' / model_name

def checkpoint_mismatches(candidate):
    expected = {'arch': APP_ARCH, 'run': 'final', 'classes': classes, 'seed': run['seed'],
                'best_epoch': run['best_epoch'], 'img_size': a_state['settings']['img_size'],
                'mean': a_state['settings']['normalisation']['mean'],
                'std': a_state['settings']['normalisation']['std']}
    different = [field for field, value in expected.items() if candidate.get(field) != value]
    if candidate.get('row', 'A') != APP_ROW:
        different.append('row')
    metrics = candidate.get('metrics', {})
    for field, value in {'validation_accuracy': run['val']['accuracy'],
                         'validation_macro_F1': run['val']['macro_f1'],
                         'test_accuracy': run['test']['accuracy'],
                         'test_macro_F1': run['test']['macro_f1']}.items():
        if field not in metrics or abs(float(metrics[field]) - value) > 1e-10:
            different.append(f'metrics.{field}')
    return different

# Never use a stale same-named model merely because its folder sorts first.
model_candidates = [preferred_model] if preferred_model.is_file() else find_files(INPUT_ROOT, model_name)
ckpt, model_file, rejected_models = None, None, []
for file in model_candidates:
    candidate = torch.load(file, map_location='cpu', weights_only=True)
    different = checkpoint_mismatches(candidate)
    if not different:
        ckpt, model_file = candidate, file
        break
    rejected_models.append(f'{file}: {", ".join(different)}')
if ckpt is None:
    detail = '; '.join(rejected_models) or f'expected {preferred_model}'
    raise ValueError(f'No checkpoint matches the selected validation-best run '
                     f'({APP_ARCH}, row {APP_ROW}, seed {run["seed"]}, epoch {run["best_epoch"]}). '
                     f'Attach matching results/ and models/ together. Details: {detail}')
print(f'Model file: {model_file} (seed {ckpt["seed"]}, best epoch {ckpt["best_epoch"]}; metadata checked)')

class CustomCNN(nn.Module):
    """Notebook 02's custom CNN (needed to load it)."""
    def __init__(self, num_classes, widths=(32, 64, 128, 256, 512)):
        super().__init__()
        layers, c_in = [], 3
        for c in widths:
            for _ in range(2):
                layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True)]
                c_in = c
            layers.append(nn.MaxPool2d(2))
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2),
                                        nn.Linear(c_in, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

def build(arch):
    if arch == 'custom_cnn':
        return CustomCNN(K)
    if arch == 'vgg16':
        net = models.vgg16(weights=None)
        net.classifier[6] = nn.Linear(net.classifier[6].in_features, K)
    elif arch == 'resnet50':
        net = models.resnet50(weights=None)
        net.fc = nn.Linear(net.fc.in_features, K)
    else:
        net = models.mobilenet_v2(weights=None)
        net.classifier[1] = nn.Linear(net.last_channel, K)
    return net

net = build(APP_ARCH)
net.load_state_dict(ckpt['state_dict'])
net = net.to(DEVICE).eval()
IMG_SIZE = int(ckpt.get('img_size', 224))
MEAN_255 = torch.tensor(ckpt['mean'], device=DEVICE).view(1, 3, 1, 1) * 255
STD_255 = torch.tensor(ckpt['std'], device=DEVICE).view(1, 3, 1, 1) * 255

# the input of the last layer = the image's features (used to pick typical library photos)
last_linear = [m for m in net.modules() if isinstance(m, nn.Linear)][-1]
feats = {}
last_linear.register_forward_hook(lambda m, inp, out: feats.__setitem__('x', inp[0].detach()))

def load_image(path):
    """As notebooks 02 and 04 and the web app: centre square, bilinear resize."""
    with Image.open(path) as img:
        img = img.convert('RGB')
        side = min(img.size)
        left, top = (img.width - side) // 2, (img.height - side) // 2
        return np.asarray(img.crop((left, top, left + side, top + side)).resize((IMG_SIZE, IMG_SIZE),
                                                                               Image.Resampling.BILINEAR))

@torch.inference_mode()
def predict(paths, batch=64):
    """Class probabilities and features for each image."""
    probs, features = [], []
    with ThreadPoolExecutor(8) as pool:
        for i in range(0, len(paths), batch):
            x = torch.from_numpy(np.stack(list(pool.map(load_image, paths[i:i + batch])))).to(DEVICE)
            x = (x.permute(0, 3, 1, 2).float() - MEAN_255) / STD_255
            probs.append(torch.softmax(net(x).float(), 1).cpu())
            features.append(feats['x'].float().cpu())
    return torch.cat(probs).numpy(), torch.cat(features).numpy()

## 3. The "not sure" threshold, from the validation split

In [ ]:
start = time.time()
val = splits[splits['split'] == 'val'].reset_index(drop=True)
val_probs, _ = predict([COLOR_DIR / p for p in val['path']])
confidence = val_probs.max(1)
correct = val_probs.argmax(1) == val['label'].to_numpy()
raw = float(np.quantile(confidence[correct], 1 - KEEP_CORRECT))
THRESHOLD = float(np.clip(np.floor(raw * 100) / 100, *THRESHOLD_LIMITS))
kept = float(np.mean(confidence[correct] >= THRESHOLD))
caught = int(np.sum(confidence[~correct] < THRESHOLD))
print(f'Validation: {len(val):,} images, accuracy {100 * correct.mean():.2f}%  ({time.time() - start:.0f} s)')
print(f'{100 * (1 - KEEP_CORRECT):.0f}% of the correct answers have a confidence below {raw:.3f} '
      f'→ threshold {THRESHOLD:.2f}' + (' (limited to the allowed range)' if THRESHOLD != np.floor(raw * 100) / 100 else ''))
print(f'With it: {100 * kept:.1f}% of the correct answers are given, and {caught} of the {int((~correct).sum())} '
      'wrong answers become "Not sure".')

fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.linspace(min(0.3, confidence.min()), 1, 36)
ax.hist(confidence[correct], bins=bins, color=BLUE, label='correct answers')
ax.hist(confidence[~correct], bins=bins, color=ORANGE, label='wrong answers')
ax.axvline(THRESHOLD, color=INK, linewidth=1.5)
ax.text(THRESHOLD, ax.get_ylim()[1] * 0.9, f'  threshold {THRESHOLD:.2f}', color=INK, fontsize=10)
ax.set_yscale('log')
ax.set_xlabel('Confidence (the highest class probability)')
ax.set_ylabel('Validation images (log scale)')
ax.legend(frameon=False, loc='upper left')
ax.set_title(f'{NAMES[APP_ARCH]}, row {APP_ROW}: confidence on the validation split')
fig.savefig(RESULTS_OUT / '05_confidence_threshold.png')
plt.show()

app_ckpt = dict(ckpt)
app_ckpt.update(
    name=f'{NAMES[APP_ARCH]} ({ROW_NAMES[APP_ROW].lower()})',
    training_images=ROW_NAMES[APP_ROW], row=APP_ROW,
    metrics={'validation_accuracy': run['val']['accuracy'], 'validation_macro_F1': run['val']['macro_f1'],
             'test_accuracy': run['test']['accuracy'], 'test_macro_F1': run['test']['macro_f1']},
    confidence_threshold=THRESHOLD,
    threshold_rule=f'{100 * KEEP_CORRECT:g}% of the correct validation answers are above it, '
                   f'limited to {THRESHOLD_LIMITS[0]}-{THRESHOLD_LIMITS[1]}',
    exported_utc=datetime.now(timezone.utc).isoformat(timespec='seconds'),
)
torch.save(app_ckpt, APP_OUT / 'models' / 'model.pt')
print(f'\nSaved app/models/model.pt ({(APP_OUT / "models" / "model.pt").stat().st_size / 1e6:.1f} MB): '
      f'{app_ckpt["name"]}, test accuracy {100 * run["test"]["accuracy"]:.2f}%')

## 4. Library photos

One training photo per class for the app's disease library: among up to 300 training photos of the class that the
app model recognises with at least 99% confidence, the one whose features are closest to the average of the class
(a typical leaf, not an odd one). Check the sheet below; to use another photo, put its path in `LIBRARY_PICKS`.

In [ ]:
train = splits[splits['split'] == 'train']
rng = np.random.default_rng(SEED)
picks, sheet = {}, []
for c in classes:
    rows = train[train['label'] == c['label']]
    rows = rows.iloc[rng.permutation(len(rows))[:LIBRARY_PER_CLASS]]
    if c['folder'] in LIBRARY_PICKS:
        picks[c['folder']] = LIBRARY_PICKS[c['folder']]
    else:
        probs, f = predict([COLOR_DIR / p for p in rows['path']])
        good = probs[:, c['label']] >= 0.99
        if not good.any():
            good = np.ones(len(rows), bool)
        f = f / np.linalg.norm(f, axis=1, keepdims=True)
        centre = f[good].mean(0)
        best = np.flatnonzero(good)[np.argmax(f[good] @ centre)]
        picks[c['folder']] = rows['path'].iloc[best]
    with Image.open(COLOR_DIR / picks[c['folder']]) as img:
        img = img.convert('RGB')
        side = min(img.size)
        left, top = (img.width - side) // 2, (img.height - side) // 2
        img = img.crop((left, top, left + side, top + side))
        img.save(APP_OUT / 'frontend/public/library' / f'{SLUGS[c["folder"]]}.jpg', quality=90)
        sheet.append((c['name'], np.asarray(img.resize((200, 200)))))

fig, axes = plt.subplots(2, 5, figsize=(14, 6.2))
for ax, (name, img) in zip(axes.flat, sheet):
    ax.imshow(img)
    ax.set_title(name, fontsize=10, fontweight='normal')
    ax.axis('off')
fig.suptitle('Library photos (from the train split)', x=0.01, ha='left', fontweight='bold')
fig.savefig(RESULTS_OUT / '05_library_photos.png')
plt.show()
pd.Series(picks, name='path').to_csv(RESULTS_OUT / '05_library_photos.csv')

## 5. GAN images for the About page, and the results table

In [ ]:
gan = None
if synth_csv is not None:
    facts = {r['class']: r for r in summary.get('per_class', [])}
    (APP_OUT / 'frontend/public/gan').mkdir(parents=True, exist_ok=True)
    gan_classes = []
    for c in sorted(classes, key=lambda c: int((train['label'] == c['label']).sum())):   # the smallest class first
        rows = synthetic[synthetic['label'] == c['label']]
        if not len(rows):
            continue
        slug = SLUGS[c['folder']]
        samples = []
        for n, p in enumerate(rows.sample(n=min(GAN_SAMPLES, len(rows)), random_state=SEED)['path'], 1):
            shutil.copy(synth_csv.parent / p, APP_OUT / 'frontend/public/gan' / f'{slug}-{n}.jpg')
            samples.append(f'/gan/{slug}-{n}.jpg')
        f = facts.get(c['name'], {})
        gan_classes.append({'id': c['folder'], 'slug': slug, 'name': c['name'],
                            'real': int((train['label'] == c['label']).sum()), 'generated': int(len(rows)),
                            'passRate': round(f['passed %'] / 100, 3) if 'passed %' in f else None,
                            'kid': round(f['KID x1000'], 2) if 'KID x1000' in f else None,
                            'kidRealVsReal': round(f['KID x1000 real vs real'], 2) if 'KID x1000 real vs real' in f else None,
                            'samples': samples})
    gan = {'model': 'StyleGAN2-ADA, class-conditional, 256 × 256 px', 'kimg': summary.get('kimg'),
           'fillTarget': summary.get('fill_target', 2000), 'labelCheck': summary.get('label_check_model', {}).get('name'),
           'classes': gan_classes}
    print(f'GAN images: {sum(len(c["samples"]) for c in gan_classes)} copied for {len(gan_classes)} classes')

js_header = '// Made by notebooks/05_app_files.ipynb; edit the notebook, not this file, if the numbers change.\n'
(APP_OUT / 'frontend/src/data/gan.js').write_text(
    js_header + '// The GAN facts and sample images shown on the About page (null: no GAN yet). KID is x 1000.\n'
    f'export const gan = {json.dumps(gan, indent=2, ensure_ascii=False)}\n', encoding='utf-8')

results_rows = []
for row in ROW_NAMES:
    for arch in MODELS:
        rs = group(row, arch)
        if len(rs) == SEEDS_PER_ROW:
            acc = [r['test']['accuracy'] for r in rs]
            f1 = [r['test']['macro_f1'] for r in rs]
            results_rows.append({'model': NAMES[arch], 'images': ROW_NAMES[row],
                                 'accuracy': round(float(np.mean(acc)), 4), 'macroF1': round(float(np.mean(f1)), 4),
                                 'accuracySd': round(float(np.std(acc, ddof=1)), 4),
                                 'macroF1Sd': round(float(np.std(f1, ddof=1)), 4)})
lines = ',\n'.join('  ' + json.dumps(r, ensure_ascii=False) for r in results_rows)
(APP_OUT / 'frontend/src/data/results.js').write_text(
    js_header +
    '// The test-split results shown on the About page: the mean (and sd) of 3 seeds per model and training set.\n'
    '// Row A comes from notebook 02, rows B-E from notebook 04.\n'
    f'export const trainingSets = {json.dumps([ROW_NAMES[r] for r in ROW_NAMES], ensure_ascii=False)}\n\n'
    f'export const results = [\n{lines},\n]\n', encoding='utf-8')
print(f'results.js: {len(results_rows)} rows')
print(pd.DataFrame(results_rows)[['images', 'model', 'accuracy', 'macroF1']].to_string(index=False))

## 6. Save

In [ ]:
info = {'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'), 'app_arch': APP_ARCH,
        'app_row': APP_ROW, 'seed': ckpt['seed'], 'model_file': str(model_file), 'threshold': THRESHOLD,
        'threshold_raw': raw, 'correct_kept': kept, 'wrong_caught': caught, 'library_photos': picks,
        'gan_classes': [c['name'] for c in gan['classes']] if gan else [],
        'best_epoch': run['best_epoch'], 'split_fingerprint': SPLIT_FINGERPRINT,
        'comparison_runs': len(runs), 'comparison_rows': len(results_rows),
        'complete_comparison': not missing, 'row_A_file': str(a_file),
        'row_B_E_file': str(b_file) if b_file else None,
        'synthetic_csv': str(synth_csv) if synth_csv else None,
        'gan_summary_file': str(gan_summary_file) if gan_summary_file else None,
        'gan_quality_facts_available': synthetic is not None and gan_summary_file is not None}
(RESULTS_OUT / '05_app_export.json').write_text(json.dumps(info, indent=1))

with zipfile.ZipFile(OUT_DIR / 'app_files.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(APP_OUT.rglob('*')):
        if f.is_file():
            z.write(f, f.relative_to(OUT_DIR))
for f in sorted(APP_OUT.rglob('*')):
    if f.is_file():
        print(f'{f.relative_to(OUT_DIR)}  ({f.stat().st_size / 1e3:,.0f} kB)')
print(f'\napp_files.zip  ({(OUT_DIR / "app_files.zip").stat().st_size / 1e6:.1f} MB)')

## Checklist and how to use the files

1. **Threshold** (`05_confidence_threshold.png`): it should sit left of almost all correct answers. A threshold at the
   upper limit (0.95) means the model is very sure on validation; the app is then strict with unclear photos.
2. **Library photos** (`05_library_photos.png`): each should be a clear example of its class. If one is not, copy a
   better path from `splits.csv` (a train image) into `LIBRARY_PICKS` and run again.
3. **Copy into the project:** unzip `app_files.zip` in the project folder (it holds `app/...`) and let it replace the
   files. Then run `run-app.bat`. The app prints which model it loaded; the About page shows its name, the threshold,
   the results table and the GAN images, and the "Demo mode" banner is gone.

4. **Final comparison:** the log must show `Comparison: 60/60 runs` and `results.js: 20 rows`.
5. **GAN facts:** confirm that a matching `03_summary.json` was found if you want the quality facts on the About page.

Download `results/` as `results-05-session2/` for review. `05_app_export.json` records the selected source files,
seed, best epoch, comparison completeness and whether GAN quality facts were available.